<a href="https://colab.research.google.com/github/vkhanht1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkhanht1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
repo_id = "FlyRank/internship-warehouse"

print("Scanning repo files...")
all_files = list_repo_files(repo_id, repo_type="dataset", token=hf_token)
parquet_files = [f for f in all_files if f.endswith(".parquet")]
gsc_files = [f for f in parquet_files if any(k in f.lower() for k in ["fact", "gsc", "performance"])]

if not gsc_files:
    gsc_files = parquet_files
target_files = [f for f in gsc_files if "2026-03" in f]
if not target_files:
    target_files = gsc_files[:3]
print(f"Target GSC files: {target_files}")

if not target_files:
    raise ValueError("Don't found any file .parquet in repo!")
local_paths = [hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", token=hf_token) for f in target_files]

con = duckdb.connect()
con.sql(f"CREATE VIEW df_raw AS SELECT * FROM read_parquet({local_paths})")

# Check Signal 1: Inspect the actual schema of df_raw
print("\n--- SCHEMA OF TABLE ---")
print(con.sql("DESCRIBE df_raw").df())

# Check Signal 2: Search Impressions Volume using 'gsc_impressions'
print("\n--- SIGNAL 2: Impression Volume ---")
s2_q = """
SELECT
    CASE
        WHEN gsc_impressions >= 1000 THEN 'High Vol (>=1k)'
        WHEN gsc_impressions >= 100 THEN 'Mid Vol (100-999)'
        ELSE 'Low Vol (<100)'
    END as vol_bucket,
    COUNT(*) as n,
    ROUND(AVG(ga4_sessions), 2) as avg_sessions
FROM df_raw
GROUP BY 1 ORDER BY 1;
"""
print(con.sql(s2_q).df())
print("Verdict Signal 2: CONFIRMED")

Scanning repo files...
Target GSC files: ['fact_content_daily_performance/month=2026-03/data_0.parquet']

--- SCHEMA OF TABLE ---
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12          

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Create the output directory if it does not exist
os.makedirs("work/outputs", exist_ok=True)

# Query to calculate the baseline score and rank rows
# Note: Combining search impressions and GA4 traffic metrics
query_rank = """
SELECT
    *,
    (COALESCE(gsc_impressions, 0) * 0.4 + COALESCE(ga4_sessions, 0) * 0.6) AS baseline_score
FROM df_raw
ORDER BY baseline_score DESC
"""

# Execute query and convert result to DataFrame
df_ranked = con.sql(query_rank).df()

# Assign explicit rank column
df_ranked['rank'] = range(1, len(df_ranked) + 1)

# Save ranked dataset to CSV
output_path = "work/outputs/baseline_action_score.csv"
df_ranked.to_csv(output_path, index=False)

print(f"Successfully exported ranked results to: {output_path}")
print(df_ranked.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully exported ranked results to: work/outputs/baseline_action_score.csv
  report_date           client_hash_id           content_hash_id  \
0  2026-03-28  client_23a62021009f63c4  content_44f34c0a90047651   
1  2026-03-29  client_e547b89c05043229  content_eadb33b5df496f4a   
2  2026-03-04  client_62f4a7e64f5e0096  content_34a70fea29d15f24   
3  2026-03-28  client_e547b89c05043229  content_eadb33b5df496f4a   
4  2026-03-04  client_62f4a7e64f5e0096  content_945d6ff91386c817   
5  2026-03-30  client_e547b89c05043229  content_eadb33b5df496f4a   
6  2026-03-27  client_e547b89c05043229  content_eadb33b5df496f4a   
7  2026-03-31  client_e547b89c05043229  content_eadb33b5df496f4a   
8  2026-03-24  client_e547b89c05043229  content_eadb33b5df496f4a   
9  2026-03-30  client_73cda7b4e4f265ea  content_fec55986a1868d62   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Extract the top 20 candidates from the ranked DataFrame
top_20 = df_ranked.head(20)

# Display Top 20 records
print("--- TOP 20 REVIEW ---")
display(top_20)

# Summary statistics for Top 20
print("\nSummary statistics of Top 20:")
print(top_20.describe())

--- TOP 20 REVIEW ---


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,baseline_score,rank
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,True,True,True,False,40084,1,3341,...,0,0,0,0,0,0,0,2026-03,16033.6,1
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,39305,252,86373,...,0,0,0,0,0,0,26,2026-03,15795.2,2
2,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,True,False,True,<NA>,39003,2,107840,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,15601.2,3
3,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,38436,271,84405,...,0,0,0,0,0,0,16,2026-03,15459.0,4
4,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,True,False,True,<NA>,37368,0,321886,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03,14947.2,5
5,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,35404,225,77478,...,0,0,0,0,0,0,13,2026-03,14224.0,6
6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,34817,223,75948,...,0,0,0,0,0,0,13,2026-03,13996.4,7
7,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,34606,235,77604,...,0,0,0,0,0,0,23,2026-03,13913.2,8
8,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,33571,215,77517,...,0,0,0,0,0,0,14,2026-03,13498.0,9
9,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,True,True,True,False,33383,0,6059,...,0,0,0,0,0,0,0,2026-03,13353.2,10



Summary statistics of Top 20:
               report_date  gsc_impressions  gsc_clicks  gsc_sum_position  \
count                   20        20.000000   20.000000         20.000000   
mean   2026-03-24 15:36:00     34173.200000  114.000000      62115.400000   
min    2026-03-04 00:00:00     30573.000000    0.000000       2625.000000   
25%    2026-03-24 00:00:00     31652.750000    1.000000       4593.000000   
50%    2026-03-26 12:00:00     33170.500000  100.000000      74975.500000   
75%    2026-03-29 00:00:00     35895.000000  221.500000      77711.000000   
max    2026-03-31 00:00:00     40084.000000  271.000000     321886.000000   
std                    NaN      3095.509763  116.831863      72167.068287   

       gsc_avg_position  ga4_pageviews  ga4_sessions  ga4_users  \
count         20.000000           18.0          18.0       18.0   
mean           1.754416      64.722222     61.444444       60.0   
min            0.083350            0.0           0.0        0.0   
25%    

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Leakage Check: Verify if any future data (e.g., month > '2026-03') leaked in
leakage_check = con.sql("""
SELECT COUNT(*) as leaked_rows
FROM df_raw
WHERE month > '2026-03'
""").df()

print("--- LEAKAGE CHECK ---")
print(f"Number of potentially leaked records: {leakage_check['leaked_rows'].iloc[0]}")

# 2. Weak Picks Check: Identify anomalies in Top 20 (e.g., high impressions but zero sessions)
weak_picks = top_20[(top_20['gsc_impressions'] > 1000) & (top_20['ga4_sessions'] == 0)]

print("\n--- WEAK PICKS IN TOP 20 ---")
if len(weak_picks) == 0:
    print("No weak picks or anomalies detected in Top 20.")
else:
    print(f"Detected {len(weak_picks)} weak pick(s):")
    display(weak_picks)

--- LEAKAGE CHECK ---
Number of potentially leaked records: 0

--- WEAK PICKS IN TOP 20 ---
Detected 3 weak pick(s):


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month,baseline_score,rank
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,True,True,True,False,40084,1,3341,...,0,0,0,0,0,0,0,2026-03,16033.6,1
9,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,True,True,True,False,33383,0,6059,...,0,0,0,0,0,0,0,2026-03,13353.2,10
15,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,True,True,True,False,31472,0,2625,...,0,0,0,0,0,0,0,2026-03,12588.8,16


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.